# Experimentos por algoritmo
Compara estructuras probabil?sticas, conjuntos, b?squeda vectorial, influencia, comunidades y estructuras base. Guarda corridas, par?metros y poblaciones reales. No llama benchmark del pipeline a crear una lista y un set.

In [ ]:
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print('Python:', sys.executable)
print('Proyecto:', Path.cwd())


## Ejecutar o reutilizar resultados
Esta celda usa resultados guardados por defecto. Para regenerar cambia `EJECUTAR` a True. Tres repeticiones solo sirven para diagn?stico; usa 100 cuando sea viable y justifica l?mites en escalas grandes.

In [ ]:
import json
from benchmarks import run_benchmarks
EJECUTAR = False
if EJECUTAR:
    informe = run_benchmarks(sizes=['1K'], repetitions=100, max_entities=1000, queries=5)
else:
    ruta = Path('output/benchmarks/report.json')
    if not ruta.exists():
        raise FileNotFoundError('Ejecuta primero benchmarks.py; no se inventan resultados.')
    informe = json.loads(ruta.read_text(encoding='utf-8'))
print(informe['platform'], informe['python'], informe['memory_scope'])


In [ ]:
import pandas as pd
filas = []
for dataset in informe['datasets']:
    for algoritmo, metricas in dataset['summary'].items():
        filas.append({'dataset':Path(dataset['path']).stem,'eventos':dataset['events'],
                      'vectores':dataset['vectors'],'conjuntos':dataset['sets'],
                      'nodos':dataset['nodes'],'aristas':dataset['edges'],
                      'algoritmo':algoritmo,'repeticiones':dataset['repetitions'],
                      'tiempo_medio_s':metricas['seconds']['mean'],
                      'desviacion_s':metricas['seconds']['stdev'],
                      'memoria_pico_bytes':metricas['peak_bytes']['mean']})
pd.DataFrame(filas)


## Calidad y l?mites
Inspecciona FPR, error de conteo, recall@5, precisi?n de LSH, modularidad y estabilidad ARI. El JSON registra recortes; 10M eventos no equivalen a 10M vectores. El grafo experimental utiliza ?rdenes simuladas, documentadas en `graph_provenance`.

In [ ]:
dataset = informe['datasets'][0]
{algoritmo:metricas for algoritmo,metricas in dataset['summary'].items()
 if algoritmo in ['bloom','cms','minhash_lsh','hnsw','pagerank','louvain','label_propagation']}


In [ ]:
from IPython.display import Image, display
ruta = Path('output/benchmarks/scalability.png')
if ruta.exists(): display(Image(filename=str(ruta)))


## Escalabilidad
Una sola escala no permite concluir una curva de crecimiento. Ejecuta las cinco escalas con el script y declara los tama?os efectivos de cada familia. Revisa `docs/FUENTES_Y_METODOLOGIA.md` antes de interpretar los resultados.